In [0]:
# 1. READ RAW DATA WITH PERMISSIVE MODE

In [0]:

raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load("/Volumes/workspace/default/poke/pokedex.csv")).show()

+---+----------+------+------+---+------+-------+--------+---------+-----+---------------+-------+--------------------+
| id|      name|height|weight| hp|attack|defense|s_attack|s_defense|speed|           type|evo_set|                info|
+---+----------+------+------+---+------+-------+--------+---------+-----+---------------+-------+--------------------+
|  1| bulbasaur|     7|    69| 45|    49|     49|      65|       65|   45| {grass,poison}|      1|A strange seed wa...|
|  2|   ivysaur|    10|   130| 60|    62|     63|      80|       80|   60| {grass,poison}|      1|When the bulb on ...|
|  3|  venusaur|    20|  1000| 80|    82|     83|     100|      100|   80| {grass,poison}|      1|The plant blooms ...|
|  4|charmander|     6|    85| 39|    52|     43|      60|       50|   65|         {fire}|      2|Obviously prefers...|
|  5|charmeleon|    11|   190| 58|    64|     58|      80|       65|   80|         {fire}|      2|When it swings it...|
|  6| charizard|    17|   905| 78|    84

In [0]:
# Print initial metadata

In [0]:
print("Column Names:", raw_df.columns)

Column Names: ['id', 'name', 'height', 'weight', 'hp', 'attack', 'defense', 's_attack', 's_defense', 'speed', 'type', 'evo_set', 'info']


In [0]:
total_rows = raw_df.count()
print("Total Rows:", total_rows)

Total Rows: 1025


In [0]:
raw_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- height: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- hp: string (nullable = true)
 |-- attack: string (nullable = true)
 |-- defense: string (nullable = true)
 |-- s_attack: string (nullable = true)
 |-- s_defense: string (nullable = true)
 |-- speed: string (nullable = true)
 |-- type: string (nullable = true)
 |-- evo_set: string (nullable = true)
 |-- info: string (nullable = true)



In [0]:
print("Number of Columns:", len(raw_df.columns))

Number of Columns: 13


In [0]:
# Check for corrupted records

In [0]:

if "_corrupt_record" in raw_df.columns:
    corrupt_df = raw_df.filter(col("_corrupt_record").isNotNull())
    corrupt_count = corrupt_df.count()
    print(f"Total Corrupted Records Found: {corrupt_count}")
    if corrupt_count > 0:
        corrupt_df.select("_corrupt_record").show(truncate=False)
else:
    print("No corrupt records column found or 0 corrupt records detected.")

No corrupt records column found or 0 corrupt records detected.


In [0]:
# 2. SCHEMA DEFINITION & CORRUPT RECORD CHECK

In [0]:
from pyspark.sql.types import * 

poke_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("height", StringType(), True),
    StructField("weight", StringType(), True),
    StructField("hp", StringType(), True),
    StructField("attack", StringType(), True),
    StructField("defense", StringType(), True),
    StructField("s_attack", StringType(), True),
    StructField("s_defense", StringType(), True),
    StructField("speed", StringType(), True),
    StructField("type", StringType(), True),
    StructField("evo_set", StringType(), True),
    StructField("info", StringType(), True),
    StructField("_corrupt_record", StringType(), True) 
])

df_schema_applied = (spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(poke_schema)
    .csv("/Volumes/workspace/default/poke/pokedex.csv")
)

In [0]:
# 3. TRANSFORMATIONS & DATA CLEANING

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_transformed = (
    df_schema_applied
    .select(
        col("id").alias("pokemon_id"),
        col("name").alias("pokemon_name"),
        col("type").alias("primary_type"),
        col("height"),
        col("weight"),
        col("hp"),
        col("attack"),
        col("defense"),
        col("speed")
    )
)

In [0]:
df_schema_applied.withColumn("data_source", lit("Kaggle_Pokedex"))

DataFrame[id: string, name: string, height: string, weight: string, hp: string, attack: string, defense: string, s_attack: string, s_defense: string, speed: string, type: string, evo_set: string, info: string, _corrupt_record: string, data_source: string]

In [0]:
df_schema_applied.withColumn("pokemon_name_upper", upper(col("pokemon_name")))

In [0]:
df_schema_applied.withColumnRenamed("primary_type", "elemental_type")

DataFrame[id: string, name: string, height: string, weight: string, hp: string, attack: string, defense: string, s_attack: string, s_defense: string, speed: string, type: string, evo_set: string, info: string, _corrupt_record: string]

In [0]:
df_schema_applied.withColumn("height_m", col("height").cast(DoubleType()) / 10)
df_schema_applied.withColumn("weight_kg", col("weight").cast(DoubleType()) / 10)
df_schema_applied.withColumn("hp", col("hp").cast(IntegerType()))
df_schema_applied.withColumn("attack", col("attack").cast(IntegerType()))
df_schema_applied.withColumn("defense", col("defense").cast(IntegerType()))
df_schema_applied.withColumn("speed", col("speed").cast(IntegerType()))

DataFrame[id: string, name: string, height: string, weight: string, hp: string, attack: string, defense: string, s_attack: string, s_defense: string, speed: int, type: string, evo_set: string, info: string, _corrupt_record: string]

In [0]:
df_schema_applied.withColumn("total_stats", col("hp") + col("attack") + col("defense") + col("speed"))

In [0]:
df_schema_applied.filter(col("total_stats") > 0)

DataFrame[id: string, name: string, height: string, weight: string, hp: string, attack: string, defense: string, s_attack: string, s_defense: string, speed: string, type: string, evo_set: string, info: string, _corrupt_record: string]

In [0]:
df_schema_applied.drop("height", "weight")

DataFrame[id: string, name: string, hp: string, attack: string, defense: string, s_attack: string, s_defense: string, speed: string, type: string, evo_set: string, info: string, _corrupt_record: string]

In [0]:
# Handle Missing Values & Deduplication

In [0]:
from pyspark.sql.functions import *

df_cleaned = (
    df_transformed
    .fillna({"elemental_type": "Unknown"})
    .dropna(subset=["pokemon_name"])
)

In [0]:
from pyspark.sql.functions import col

df_cleaned = (
    df_transformed
    .fillna({"primary_type": "Unknown"})
    .dropna(subset=["pokemon_name"])
)

In [0]:
df_deduped = df_cleaned.dropDuplicates()
print(f"Total rows after deduplication: {df_deduped.count()}")

Total rows after deduplication: 1025


In [0]:
#4. SAVE PROCESSED DATA TO PARQUET

In [0]:
output_path = "/Volumes/workspace/default/poke/processed_pokemon_parquet"

(df_deduped.write
    .mode("overwrite")
    .format("parquet")
    .save(output_path)
)

print(f"Successfully saved transformed dataset to {output_path}")

Successfully saved transformed dataset to /Volumes/workspace/default/poke/processed_pokemon_parquet
